<a href="https://colab.research.google.com/github/donlap/stat424/blob/main/Applications_of_Optimal_Transport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pot matplotlib scikit-image scipy scikit-learn pyvista plotly

import io
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import numpy as np
import ot
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyvista as pv
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
import urllib.request

## Color transfer

In [ ]:
url_source = "https://raw.githubusercontent.com/jrosebr1/color_transfer/refs/heads/master/images/ocean_day.jpg"
url_target = "https://raw.githubusercontent.com/jrosebr1/color_transfer/refs/heads/master/images/ocean_sunset.jpg"

req_source = urllib.request.urlopen(url_source).read()
req_target = urllib.request.urlopen(url_target).read()

# Load as RGB and scale to [0, 1]
img_source = np.array(Image.open(io.BytesIO(req_source)).convert('RGB'))[::3, ::3] / 255.0
img_target = np.array(Image.open(io.BytesIO(req_target)).convert('RGB'))[::3, ::3] / 255.0

In [ ]:
h_s, w_s, _ = img_source.shape
pixels_source = img_source.reshape(-1, 3)
pixels_target = img_target.reshape(-1, 3)

# Extract Color Palettes
print("Extracting color palettes...")
kmeans_s = KMeans(n_clusters=300, random_state=42, n_init=3).fit(pixels_source)
kmeans_t = KMeans(n_clusters=300, random_state=42, n_init=3).fit(pixels_target)

palette_source = kmeans_s.cluster_centers_
palette_target = kmeans_t.cluster_centers_

### Animation of Sinkhorn iteration

In [ ]:
# @title
def sinkhorn_all_iters(a, b, M, reg, num_iter=100):
    K = np.exp(-M / reg)
    u = np.ones_like(a) / len(a)
    v = np.ones_like(b) / len(b)

    couplings = []
    for i in range(num_iter):
        u = a / (K @ v + 1e-16)
        v = b / (K.T @ u + 1e-16)

        P = np.diag(u) @ K @ np.diag(v)
        couplings.append(P)

    return couplings

print("Calculating optimal transport...")

couplings_color = sinkhorn_all_iters(a, b, M_color, reg=0.01, num_iter=45)

# Setup Matplotlib Grid and Static Target Data
print("Generating Matplotlib animation...")
fig = plt.figure(figsize=(14, 9))
fig.suptitle("Optimal Transport: Algorithm Convergence (Sinkhorn Iterations)", fontsize=16)

gs_master = fig.add_gridspec(1, 2, width_ratios=[1.2, 1], wspace=0.2)

gs_left = gs_master[0].subgridspec(2, 1, hspace=0.1)
ax_img = fig.add_subplot(gs_left[0])
ax_target = fig.add_subplot(gs_left[1])

gs_right = gs_master[1].subgridspec(3, 1, hspace=0.4)
ax_r = fig.add_subplot(gs_right[0])
ax_g = fig.add_subplot(gs_right[1])
ax_b = fig.add_subplot(gs_right[2])

bins = np.linspace(0, 1, 64)
bin_centers = 0.5 * (bins[1:] + bins[:-1])

target_r, _ = np.histogram(img_target[:,:,0], bins=bins)
target_g, _ = np.histogram(img_target[:,:,1], bins=bins)
target_b, _ = np.histogram(img_target[:,:,2], bins=bins)

max_y = max(target_r.max(), target_g.max(), target_b.max()) * 1.2

ax_img.axis('off')
im_display = ax_img.imshow(img_source)
ax_img.set_title("Morphing Image (Algorithm Iterations)", fontsize=14)

ax_target.axis('off')
ax_target.imshow(img_target)
ax_target.set_title("Target Image", fontsize=14)

axes_rgb = [(ax_r, target_r, 'Red', '#ff4b4b'),
            (ax_g, target_g, 'Green', '#00cc99'),
            (ax_b, target_b, 'Blue', '#4b8bff')]

lines_transported = []
for ax, target_data, name, color in axes_rgb:
    ax.fill_between(bin_centers, target_data, color=color, alpha=0.2)
    ax.plot(bin_centers, target_data, color=color, linestyle='--', alpha=0.6, label=f'Target {name}')

    line, = ax.plot([], [], color=color, linewidth=2.5, label=f'Transported {name}')
    lines_transported.append(line)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, max_y)
    ax.set_ylabel("Pixels")
    ax.legend(loc='upper right', fontsize=8)
    if ax != ax_b:
        ax.set_xticks([])
    else:
        ax.set_xlabel("Color Intensity (0 to 1)")

plt.close()

# 5. Algorithm Iteration Update Function
def update(frame):
    # Force the first frame to be the pure original image to avoid the "muddy" start
    if frame == 0:
        rec_img = img_source
    else:
        # Step through the calculated transport matrices
        P = couplings_color[frame - 1]

        # Barycentric mapping
        mapped_palette = np.diag(1/a) @ P @ palette_target

        # Rebuild image
        rec_pixels = mapped_palette[kmeans_s.labels_]
        rec_img = rec_pixels.reshape(h_s, w_s, 3)
        rec_img = np.clip(rec_img, 0, 1)

    im_display.set_data(rec_img)

    # Update histograms
    trans_r, _ = np.histogram(rec_img[:,:,0], bins=bins)
    trans_g, _ = np.histogram(rec_img[:,:,1], bins=bins)
    trans_b, _ = np.histogram(rec_img[:,:,2], bins=bins)

    lines_transported[0].set_data(bin_centers, trans_r)
    lines_transported[1].set_data(bin_centers, trans_g)
    lines_transported[2].set_data(bin_centers, trans_b)

    return [im_display, lines_transported[0], lines_transported[1], lines_transported[2]]

# We add +1 to the frames to account for our injected original frame (frame 0)
anim = FuncAnimation(fig, update, frames=len(couplings_color) + 1, interval=120, blit=True)
HTML(anim.to_jshtml())

## Croissant Distribution Plan

In [ ]:
# 1. Fetch the Manhattan Dataset directly from the POT GitHub repo
print("Downloading Manhattan dataset...")
url = "https://raw.githubusercontent.com/PythonOT/POT/master/data/manhattan.npz"
response = urllib.request.urlopen(url)
data = np.load(io.BytesIO(response.read()))

bakery_pos = data["bakery_pos"]
bakery_prod = data["bakery_prod"]
cafe_pos = data["cafe_pos"]
cafe_prod = data["cafe_prod"]
Imap = data["Imap"]

total_croissants = cafe_prod.sum()

plt.imshow(Imap, interpolation="bilinear")
plt.scatter(bakery_pos[:, 0], bakery_pos[:, 1], s=bakery_prod*2, c="r", ec="k", label="Bakeries (Supply)")
plt.scatter(cafe_pos[:, 0], cafe_pos[:, 1], s=cafe_prod*2, c="b", ec="k", label="Cafés (Demand)")

### Transport animation

In [ ]:
# @title
# We use reg=0.01 and num_iter=100 for a fairly sharp transport plan
couplings_supply = sinkhorn_all_iters(a, b, M_supply, reg=0.001, num_iter=100)

# Extract the final converged transport matrix
P_final = couplings_supply[-1]

# 3. Setup Routes for McCann Interpolation
routes = []
# Only create routes for meaningful transport values
for i in range(len(a)):
    for j in range(len(b)):
        if P_final[i, j] > 1e-4:
            routes.append({
                'start': bakery_pos[i],
                'end': cafe_pos[j],
                'mass': P_final[i, j]
            })

# 4. Setup Matplotlib Dashboard
print("Generating McCann Displacement animation...")
fig = plt.figure(figsize=(15, 6))
fig.suptitle("Optimal Transport: Logistics Routing (McCann Interpolation)", fontsize=16)

gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 1], wspace=0.1)
ax_map = fig.add_subplot(gs[0])
ax_mat = fig.add_subplot(gs[1])

# --- Format Map (Left) ---
ax_map.imshow(Imap, interpolation="bilinear")
ax_map.scatter(bakery_pos[:, 0], bakery_pos[:, 1], s=bakery_prod*2, c="r", ec="k", label="Bakeries (Supply)")
ax_map.scatter(cafe_pos[:, 0], cafe_pos[:, 1], s=cafe_prod*2, c="b", ec="k", label="Cafés (Demand)")

# Label the nodes
for i, pos in enumerate(bakery_pos):
    ax_map.text(pos[0], pos[1], str(i), color="w", fontsize=9, ha="center", va="center", fontweight="bold")
for i, pos in enumerate(cafe_pos):
    ax_map.text(pos[0], pos[1], str(i), color="w", fontsize=9, ha="center", va="center", fontweight="bold")

# Draw the background tracks based on the final transport matrix
for r in routes:
    ax_map.plot([r['start'][0], r['end'][0]],
                [r['start'][1], r['end'][1]],
                "-k", alpha=0.3, lw=r['mass'] * 100)

ax_map.axis('off')
ax_map.legend(loc="upper left")
ax_map.set_title("Goods in Transit", fontsize=14)

# Initialize moving goods (Scatter plot)
moving_goods = ax_map.scatter([], [], s=[], c='gold', ec='k', zorder=5)

# --- Format Transport Matrix (Right) ---
# We display the matrix scaled back to the actual number of croissants
im = ax_mat.imshow(P_final * total_croissants, cmap="coolwarm")
ax_mat.set_title("Final Transport Matrix (Croissants Delivered)", fontsize=14)
ax_mat.set_xlabel("Cafés (Demand)")
ax_mat.set_ylabel("Bakeries (Supply)")
ax_mat.set_xticks(range(len(b)))
ax_mat.set_yticks(range(len(a)))

# Overlay the delivery numbers inside the matrix cells
for i in range(len(a)):
    for j in range(len(b)):
        val = P_final[i, j] * total_croissants
        if val > 0.5: # Only show significant deliveries
            ax_mat.text(j, i, f"{val:.1f}", ha="center", va="center",
                        color="w" if val > 20 else "k", fontsize=10)

plt.close() # Prevent static double plot

# 5. McCann Animation Function
num_frames = 60

def update(frame):
    # Alpha goes smoothly from 0.0 (Bakeries) to 1.0 (Cafés)
    alpha = frame / (num_frames - 1)

    current_positions = []
    sizes = []

    # Calculate current position of each delivery batch
    for r in routes:
        pos = (1 - alpha) * r['start'] + alpha * r['end']
        current_positions.append(pos)
        # Scale the size of the moving dot based on the mass being transported
        sizes.append(r['mass'] * 2000)

    # Update scatter plot
    if current_positions:
        moving_goods.set_offsets(current_positions)
        moving_goods.set_sizes(sizes)

    return [moving_goods]

anim = FuncAnimation(fig, update, frames=num_frames, interval=80, blit=True)
HTML(anim.to_jshtml())

## Domain Adaptation

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# 1. Fetch Real Cross-Hospital Datasets (Cleveland vs Hungarian)
print("Loading UCI Heart Disease datasets (Cross-Hospital Domain Shift)...")
url_cleveland = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
url_hungarian = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data"

cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']

# na_values='?' handles the missing data automatically
df_source = pd.read_csv(url_cleveland, names=cols, na_values='?')
df_target = pd.read_csv(url_hungarian, names=cols, na_values='?')

# Clean data: Impute missing values with medians to recover structure
df_source = df_source.fillna(df_source.median(numeric_only=True))
df_target = df_target.fillna(df_target.median(numeric_only=True))

# Binarize target (0: Healthy, 1-4: Heart Disease)
y_source_raw = (df_source['target'] > 0).astype(int).values
y_target_raw = (df_target['target'] > 0).astype(int).values

X_source_raw = df_source.drop('target', axis=1).values
X_target_raw = df_target.drop('target', axis=1).values

# Equalize sample sizes to 250 for clean 1-to-1 OT visualization
np.random.seed(42)
n_samples = 250
idx_s = np.random.choice(len(X_source_raw), n_samples, replace=False)
idx_t = np.random.choice(len(X_target_raw), n_samples, replace=False)

X_source_sub = X_source_raw[idx_s]
y_source = y_source_raw[idx_s]

X_target_sub = X_target_raw[idx_t]
y_target = y_target_raw[idx_t]

# Scale features to simulate a standard ML pipeline
scaler = StandardScaler()
X_all = scaler.fit_transform(np.vstack((X_source_sub, X_target_sub)))
X_source = X_all[:n_samples]
X_target = X_all[n_samples:]

### Transport animation

In [ ]:
# @title
couplings_da = sinkhorn_all_iters(a, b, M_da, reg=0.01, num_iter=100)
P_final = couplings_da[-1]

# Calculate the final adapted coordinates
X_adapted_final = np.diag(1/a) @ P_final @ X_target

# 3. Interpolation & Machine Learning Evaluation
print("Evaluating Logistic Regression accuracy...")
clf = LogisticRegression(random_state=42, C=1.0, max_iter=500)
num_frames = 60
frames_X_13D = []
accuracies = []

for i in range(num_frames):
    alpha = i / (num_frames - 1)
    # McCann Interpolation in 13D
    X_current_13D = (1 - alpha) * X_source + alpha * X_adapted_final
    frames_X_13D.append(X_current_13D)

    # Train model on the currently adapting data
    clf.fit(X_current_13D, y_source)

    # Test model directly on the actual Target Domain (Hungarian patients)
    acc = clf.score(X_target, y_target) * 100
    accuracies.append(acc)

# 4. PCA Projection for 2D Visualization
print("Projecting frames to 2D for visualization...")
pca = PCA(n_components=2)
X_target_2d = pca.fit_transform(X_target)
frames_X_2d = [pca.transform(frame) for frame in frames_X_13D]

# 5. Matplotlib Animation Dashboard Setup
fig, (ax_scatter, ax_acc) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Real-World Domain Adaptation: Cleveland to Hungarian Hospitals", fontsize=16)

# --- Format Scatter Plot ---
colors = ['#00cc99', '#ff4b4b'] # Healthy (Green) vs Disease (Red)
cmap_source = [colors[y] for y in y_source]

# Target Domain (Hungarian) plotted as unlabeled grey ghosts
ax_scatter.scatter(X_target_2d[:, 0], X_target_2d[:, 1], c='lightgray', marker='X', s=50, alpha=0.6, label='Hungarian Data (Unlabeled Target)')

# Moving Source Domain (Cleveland) mapped to actual labels
scatter_source = ax_scatter.scatter(frames_X_2d[0][:, 0], frames_X_2d[0][:, 1], c=cmap_source, s=50, ec='k', label='Cleveland Data (Adapting)')

ax_scatter.set_title("13D Feature Alignment (PCA 2D Projection)", fontsize=14)
ax_scatter.legend(loc='upper right')

all_x = np.concatenate([f[:, 0] for f in frames_X_2d] + [X_target_2d[:, 0]])
all_y = np.concatenate([f[:, 1] for f in frames_X_2d] + [X_target_2d[:, 1]])
ax_scatter.set_xlim(all_x.min() - 0.5, all_x.max() + 0.5)
ax_scatter.set_ylim(all_y.min() - 0.5, all_y.max() + 0.5)
ax_scatter.grid(True, linestyle='--', alpha=0.5)

# --- Format Accuracy Plot ---
ax_acc.set_title("Model Accuracy on Hungarian Patients", fontsize=14)
ax_acc.set_xlim(0, num_frames - 1)
# Dynamic Y-axis
min_acc = min(accuracies) - 2
max_acc = max(accuracies) + 2
ax_acc.set_ylim(min_acc, max_acc)
ax_acc.set_xlabel("McCann Interpolation Step")
ax_acc.set_ylabel("Accuracy (%)")
ax_acc.grid(True, linestyle='--', alpha=0.5)

line_acc, = ax_acc.plot([], [], lw=3, color='#4b8bff')

plt.close()

# 6. Animation Update Function
def update(frame):
    scatter_source.set_offsets(frames_X_2d[frame])

    x_data = list(range(frame + 1))
    y_data = accuracies[:frame + 1]
    line_acc.set_data(x_data, y_data)

    for c in list(ax_acc.collections):
        c.remove()
    ax_acc.fill_between(x_data, y_data, color='#4b8bff', alpha=0.2)

    return [scatter_source, line_acc]

anim_da = FuncAnimation(fig, update, frames=num_frames, interval=80, blit=False)
HTML(anim_da.to_jshtml())